# build_split_v2 test-set and folds

Runs **exactly once**. Generates:

- `splits/dev_pool.csv`      80% of the corpus, foundation for HPO and CV
- `splits/dev_folds.csv`     Fold number for each Dev row (0..4)
- `splits/test_LOCKED.csv`   20%, locked away until `test_evaluation_v2`
- `splits/split_manifest.json`   Seeds, SHA-256, row counts
- `splits/split_distribution.csv`  label x category per split

All splits are **group-aware** (identical texts stay together; otherwise
the corpus leaks across its ~1,500 duplicate rows) and **stratified** by
label x category.

After execution, `test_LOCKED.csv` must only be read by `test_evaluation_v2`.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import sys, os, json, datetime
# --- locate the project root -------------------------------------------
# No hardcoded Drive path: take KUSA_ROOT if it is set, otherwise the first
# candidate that actually contains config.py. Works in Colab and locally.
import os, sys
_CANDIDATES = [
    os.environ.get("KUSA_ROOT", ""),
    "/content/drive/MyDrive/kusa",
    "/content/drive/MyDrive/v2_heldout",
    "/content/drive/MyDrive/google_colab/kusa/v2_heldout",
    os.getcwd(),
    os.path.dirname(os.getcwd()),
]
V2_ROOT = next((p for p in _CANDIDATES
                if p and os.path.isfile(os.path.join(p, "config.py"))), None)
assert V2_ROOT, ("config.py not found - set KUSA_ROOT to the project "
                 "directory, e.g. os.environ['KUSA_ROOT'] = '/content/drive/MyDrive/kusa'")
sys.path.insert(0, V2_ROOT)
print("project root:", V2_ROOT)
from config import *
import utils_split as u

import pandas as pd
import numpy as np

print("Dev Pool  ->", DEV_POOL)
print("Test      ->", TEST_LOCKED)
print("Seeds     -> split=%d  hpo=%d  train=%d" % (SPLIT_SEED, HPO_SEED, TRAIN_SEED))
print("Proportions -> test=%.2f  hpo=%.2f  folds=%d" % (TEST_FRAC, HPO_FRAC, N_FOLDS))

Mounted at /content/drive
project root: /content/drive/MyDrive/kusa
Dev Pool  -> /content/drive/MyDrive/kusa/splits/dev_pool.csv
Test      -> /content/drive/MyDrive/kusa/splits/test_LOCKED.csv
Seeds     -> split=2026  hpo=2026  train=42
Proportions -> test=0.20  hpo=0.20  folds=5


## Overwrite Protection

A second run would generate a different split and invalidate all previously
trained models. Set `FORCE = True` to deliberately re-generate.

In [2]:
FORCE = False

existing = [p for p in (DEV_POOL, DEV_FOLDS, TEST_LOCKED, MANIFEST) if os.path.exists(p)]
if existing and not FORCE:
    print("Split already exists:")
    for p in existing:
        print("   ", p.replace(ROOT, "."))
    raise SystemExit("Aborted. Set FORCE = True to re-generate.")
print("No existing split found - proceeding.")

No existing split found - proceeding.


## Load Data

Rows without `surface` **or** without `lemma` are discarded. This ensures that
the baseline and both Dual-View variants operate on identical rows

In [3]:
df = pd.read_csv(PREPROCESSED, encoding="utf-8")
df = df.loc[:, ~df.columns.str.contains("^Unnamed")]
n_raw = len(df)

df = df.dropna(subset=["surface", "lemma"]).reset_index(drop=True)
df["row_id"] = np.arange(len(df))          # stable key for dev_folds.csv

print(f"Loaded: {n_raw}  |  After dropna(surface, lemma): {len(df)}"
      f"  (discarded: {n_raw - len(df)})")
print("Columns:", list(df.columns))
print()
print("Label distribution:", df["label"].value_counts().sort_index().to_dict())
print("Categories:        ", df["category"].value_counts().to_dict())

Loaded: 12306  |  After dropna(surface, lemma): 12306  (discarded: 0)
Columns: ['num_label', 'category', 'text', 'label', 'surface', 'lemma', 'row_id']

Label distribution: {0: 4102, 1: 4102, 2: 4102}
Categories:         {'social': 5463, 'news': 4302, 'art': 1759, 'health': 624, 'technology': 158}


In [4]:
# group = normalized text (identical texts stay in the same split)
# stratum = label x category
df = u.add_group_and_stratum(df)

n_groups = df["group"].nunique()
print(f"Rows: {len(df)}  |  Unique groups: {n_groups}"
      f"  |  Duplicate rows: {len(df) - n_groups}")

# Warning for groups with conflicting labels: they inevitably end up
# in the same split, but cannot be solved by any model.
conflict = df.groupby("group")["label"].nunique()
n_conf = int((conflict > 1).sum())
print(f"Groups with conflicting labels: {n_conf}")

Rows: 12306  |  Unique groups: 11481  |  Duplicate rows: 825
Groups with conflicting labels: 8


## Split 1: Separate Test Set

In [5]:
dev, test = u.grouped_holdout(df, TEST_FRAC, SPLIT_SEED)
dev  = dev.reset_index(drop=True)
test = test.reset_index(drop=True)

print(f"Dev Pool: {len(dev):5d}  ({len(dev)/len(df):.1%})")
print(f"Test:     {len(test):5d}  ({len(test)/len(df):.1%})")
assert u.no_group_overlap(dev, test), "Group overlap between Dev and Test"
print("OK - no group overlap.")

Dev Pool:  9851  (80.1%)
Test:      2455  (19.9%)
OK - no group overlap.


## Split 2: Fold Assignment in Dev Pool

Defined once here and saved to file so all three training notebooks are guaranteed to use the exact same folds.

In [6]:
dev["fold"] = u.assign_folds(dev, N_FOLDS, SPLIT_SEED).values

sizes = dev["fold"].value_counts().sort_index()
print("Fold sizes:", sizes.tolist())

for k in range(N_FOLDS):
    tr = dev[dev["fold"] != k]
    va = dev[dev["fold"] == k]
    assert u.no_group_overlap(tr, va), f"Group overlap in Fold {k}"
print("OK - no group overlap within folds.")

print()
print("Label distribution per fold:")
print(pd.crosstab(dev["fold"], dev["label"]))

Fold sizes: [1949, 1986, 1962, 1970, 1984]
OK - no group overlap within folds.

Label distribution per fold:
label    0    1    2
fold                
0      648  646  655
1      680  647  659
2      659  646  657
3      646  659  665
4      636  685  663


## Verification

In [7]:
ok = True

# 1 - Row totals
if len(dev) + len(test) != len(df):
    print("ERROR: Row totals do not match"); ok = False

# 2 - No duplicate row_id
if set(dev["row_id"]) & set(test["row_id"]):
    print("ERROR: row_id present in both splits"); ok = False

# 3 - All strata represented in test
missing = set(df["stratum"]) - set(test["stratum"])
if missing:
    print(f"NOTE: {len(missing)} strata missing in test:", sorted(missing)[:5])

# 4 - Distribution drift Dev vs Test
p_dev  = dev["label"].value_counts(normalize=True).sort_index()
p_test = test["label"].value_counts(normalize=True).sort_index()
drift = (p_dev - p_test).abs().max()
print(f"Max label share difference Dev vs Test: {drift:.4f}")
if drift > 0.02:
    print("NOTE: Difference > 2 pp - stratification disrupted by grouping constraint")

print("\nVerification:", "OK" if ok else "FAILED")
assert ok

Max label share difference Dev vs Test: 0.0078

Verification: OK


## Save

In [8]:
drop_cols = ["stratum"]        # group is kept, needed later for HPO slice

dev.drop(columns=drop_cols).to_csv(DEV_POOL, index=False, encoding="utf-8")
test.drop(columns=drop_cols).to_csv(TEST_LOCKED, index=False, encoding="utf-8")
dev[["row_id", "fold"]].to_csv(DEV_FOLDS, index=False, encoding="utf-8")

dist = pd.concat({
    "dev":  u.distribution(dev),
    "test": u.distribution(test),
}, names=["split"])
dist.to_csv(DISTRIB, encoding="utf-8")

manifest = {
    "created":            datetime.datetime.now().isoformat(timespec="seconds"),
    "source":             PREPROCESSED,
    "source_sha256":      u.sha256(PREPROCESSED),
    "n_rows_source":      int(n_raw),
    "n_rows_used":        int(len(df)),
    "n_groups":           int(n_groups),
    "n_conflicting_groups": int(n_conf),
    "split_seed":         SPLIT_SEED,
    "test_frac":          TEST_FRAC,
    "n_folds":            N_FOLDS,
    "n_dev":              int(len(dev)),
    "n_test":             int(len(test)),
    "fold_sizes":         sizes.tolist(),
    "dev_pool_sha256":    u.sha256(DEV_POOL),
    "test_locked_sha256": u.sha256(TEST_LOCKED),
    "dev_folds_sha256":   u.sha256(DEV_FOLDS),
}
with open(MANIFEST, "w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=2, ensure_ascii=False)

print(json.dumps(manifest, indent=2, ensure_ascii=False))

{
  "created": "2026-08-11T00:23:51",
  "source": "/content/drive/MyDrive/kusa/datasets/KurdiSent_preprocessed.csv",
  "source_sha256": "1ed2d675068f4d524441822b0e392596f1349e2bb9c56c65aef082cfedcdc2b6",
  "n_rows_source": 12306,
  "n_rows_used": 12306,
  "n_groups": 11481,
  "n_conflicting_groups": 8,
  "split_seed": 2026,
  "test_frac": 0.2,
  "n_folds": 5,
  "n_dev": 9851,
  "n_test": 2455,
  "fold_sizes": [
    1949,
    1986,
    1962,
    1970,
    1984
  ],
  "dev_pool_sha256": "fd949c83b3b5eca6cc422e515c8417c6f7d9e2cdf66e5fe88392da5be9d3cd5e",
  "test_locked_sha256": "fbf57402ef9aa70cec958fb55916fe70478ed85b3b732747bd68a1e3d27008d5",
  "dev_folds_sha256": "7f1025af79aeb7105a434d0888359eb915b18f2bbab77852382744a7775bb1ea"
}


## Distribution Table for Appendix

In [9]:
print("DEV POOL")
print(u.distribution(dev))
print()
print("TEST (locked from here)")
print(u.distribution(test))
print()
print("Test SHA-256:", manifest["test_locked_sha256"])
print()
print("Next step: hpo_kusa_baseline_v2")

DEV POOL
category   art  health  news  social  technology   All
label                                                 
0          435      31  1636    1117          50  3269
1          451     364  1171    1245          52  3283
2          522      94   640    2010          33  3299
All       1408     489  3447    4372         135  9851

TEST (locked from here)
category  art  health  news  social  technology   All
label                                                
0         113       6   421     291           2   833
1          95     107   278     324          15   819
2         143      22   156     476           6   803
All       351     135   855    1091          23  2455

Test SHA-256: fbf57402ef9aa70cec958fb55916fe70478ed85b3b732747bd68a1e3d27008d5

Next step: hpo_kusa_baseline_v2
